In [ ]:
!pip install transformers torch scikit-learn pandas --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [ ]:
DATASET_PATH = "/content/drive/MyDrive/Inzira Project/full_dataset.json"

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f"✓ Dataset loaded: {len(df)} total pages")
print(f"  Label 1 (opportunity)     : {len(df[df['bert_label'] == 1])}")
print(f"  Label 0 (not opportunity) : {len(df[df['bert_label'] == 0])}")

✓ Dataset loaded: 551 total pages
  Label 1 (opportunity)     : 393
  Label 0 (not opportunity) : 158


In [ ]:
df['text'] = df['text'].fillna('').astype(str)
df['text'] = df['text'].apply(lambda x: ' '.join(x.split()[:400]))

texts  = df['text'].tolist()
labels = df['bert_label'].tolist()

print(f"✓ Text prepared")
print(f"  Sample: {texts[0][:100]}...")

✓ Text prepared
  Sample: Rwanda - Banking System | export.gov Skip to content Rwanda - Banking SystemRwanda - Banking System ...


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.30, random_state=42, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"✓ Dataset split:")
print(f"  Training set   : {len(X_train)} pages")
print(f"  Validation set : {len(X_val)} pages")
print(f"  Test set       : {len(X_test)} pages")

✓ Dataset split:
  Training set   : 385 pages
  Validation set : 83 pages
  Test set       : 83 pages


In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print("✓ BERT tokenizer loaded")

class OpportunityDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = OpportunityDataset(X_train, y_train, tokenizer)
val_dataset   = OpportunityDataset(X_val,   y_val,   tokenizer)
test_dataset  = OpportunityDataset(X_test,  y_test,  tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False)

print("✓ Data loaders ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

✓ BERT tokenizer loaded
✓ Data loaders ready


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using device: {device}")

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)
model = model.to(device)
print("✓ BERT model loaded")

✓ Using device: cuda


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ BERT model loaded


In [ ]:
EPOCHS        = 5
LEARNING_RATE = 2e-5

optimizer     = AdamW(model.parameters(), lr=LEARNING_RATE, eps=1e-8)
total_steps   = len(train_loader) * EPOCHS
scheduler     = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

print(f"✓ Optimizer ready — training for {EPOCHS} epochs")

✓ Optimizer ready — training for 5 epochs


In [ ]:
def evaluate(model, data_loader, device):
    model.eval()
    all_preds  = []
    all_labels = []
    total_loss = 0

    with torch.no_grad():
        for batch in data_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels_batch   = batch['label'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels_batch
            )
            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels_batch.cpu().numpy())

    avg_loss = total_loss / len(data_loader)
    accuracy = accuracy_score(all_labels, all_preds)
    return avg_loss, accuracy, all_preds, all_labels


print("── STARTING TRAINING ───────────────────────────────────")
best_val_accuracy = 0

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels_batch   = batch['label'].to(device)

        model.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels_batch
        )
        loss = outputs.loss
        total_train_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        if batch_idx % 10 == 0:
            print(f"  Epoch {epoch+1}/{EPOCHS} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_train_loss = total_train_loss / len(train_loader)
    val_loss, val_accuracy, _, _ = evaluate(model, val_loader, device)

    print(f"\n  Epoch {epoch+1} summary:")
    print(f"  Training loss   : {avg_train_loss:.4f}")
    print(f"  Validation loss : {val_loss:.4f}")
    print(f"  Validation acc  : {val_accuracy*100:.2f}%")

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        import os
        os.makedirs("/content/drive/MyDrive/Inzira Project/models/bert_classifier", exist_ok=True)
        model.save_pretrained("/content/drive/MyDrive/Inzira Project/models/bert_classifier")
        tokenizer.save_pretrained("/content/drive/MyDrive/Inzira Project/models/bert_classifier")
        print(f"  ✓ Best model saved")

── STARTING TRAINING ───────────────────────────────────
  Epoch 1/5 | Batch 0/25 | Loss: 0.6028
  Epoch 1/5 | Batch 10/25 | Loss: 0.2833
  Epoch 1/5 | Batch 20/25 | Loss: 0.3260

  Epoch 1 summary:
  Training loss   : 0.5092
  Validation loss : 0.3581
  Validation acc  : 85.54%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved
  Epoch 2/5 | Batch 0/25 | Loss: 0.3300
  Epoch 2/5 | Batch 10/25 | Loss: 0.1561
  Epoch 2/5 | Batch 20/25 | Loss: 0.0958

  Epoch 2 summary:
  Training loss   : 0.2370
  Validation loss : 0.2145
  Validation acc  : 91.57%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved
  Epoch 3/5 | Batch 0/25 | Loss: 0.1056
  Epoch 3/5 | Batch 10/25 | Loss: 0.2036
  Epoch 3/5 | Batch 20/25 | Loss: 0.1104

  Epoch 3 summary:
  Training loss   : 0.1334
  Validation loss : 0.2082
  Validation acc  : 91.57%
  Epoch 4/5 | Batch 0/25 | Loss: 0.0503
  Epoch 4/5 | Batch 10/25 | Loss: 0.0453
  Epoch 4/5 | Batch 20/25 | Loss: 0.0400

  Epoch 4 summary:
  Training loss   : 0.0743
  Validation loss : 0.1562
  Validation acc  : 93.98%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved
  Epoch 5/5 | Batch 0/25 | Loss: 0.1784
  Epoch 5/5 | Batch 10/25 | Loss: 0.0253
  Epoch 5/5 | Batch 20/25 | Loss: 0.0774

  Epoch 5 summary:
  Training loss   : 0.0494
  Validation loss : 0.1759
  Validation acc  : 91.57%


In [ ]:
print("── FINAL TEST SET RESULTS ──────────────────────────────")
test_loss, test_accuracy, test_preds, test_labels = evaluate(model, test_loader, device)

precision = precision_score(test_labels, test_preds)
recall    = recall_score(test_labels, test_preds)
f1        = f1_score(test_labels, test_preds)

print(f"\n  Test Accuracy  : {test_accuracy*100:.2f}%")
print(f"  Precision      : {precision:.4f}")
print(f"  Recall         : {recall:.4f}")
print(f"  F1 Score       : {f1:.4f}")
print(f"\n  Target: Accuracy ≥ 82%")

if test_accuracy >= 0.82:
    print("  ✓ TARGET MET — BERT classifier is ready")
else:
    print("  ✗ Target not met")

print("\n── FULL CLASSIFICATION REPORT ──────────────────────────")
print(classification_report(
    test_labels, test_preds,
    target_names=['Not Opportunity', 'Opportunity']
))

── FINAL TEST SET RESULTS ──────────────────────────────

  Test Accuracy  : 81.93%
  Precision      : 0.8548
  Recall         : 0.8983
  F1 Score       : 0.8760

  Target: Accuracy ≥ 82%
  ✗ Target not met

── FULL CLASSIFICATION REPORT ──────────────────────────
                 precision    recall  f1-score   support

Not Opportunity       0.71      0.62      0.67        24
    Opportunity       0.85      0.90      0.88        59

       accuracy                           0.82        83
      macro avg       0.78      0.76      0.77        83
   weighted avg       0.81      0.82      0.82        83



In [ ]:
import os

os.makedirs("/content/drive/MyDrive/Inzira Project/models/bert_classifier", exist_ok=True)
model.save_pretrained("/content/drive/MyDrive/Inzira Project/models/bert_classifier")
tokenizer.save_pretrained("/content/drive/MyDrive/Inzira Project/models/bert_classifier")

print("✓ BERT model saved to Google Drive")
print("✓ Location: Inzira Project/models/bert_classifier")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ BERT model saved to Google Drive
✓ Location: Inzira Project/models/bert_classifier
